In [3]:
import os
import sys
sys.path.append(os.path.abspath(".."))

In [4]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.bm25 import BM25Retriever
from app.retrieval.faiss_retriever import FaissRetriever
from app.conversation.recommender import LLMRecommender

In [5]:
df = pd.read_csv("../data/processed/catalog_processed.csv")
embeddings = np.load(
    "../data/embeddings/catalog_embeddings.npy"
)
embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
bm25 = BM25Retriever(df)
faiss = FaissRetriever(
    df,
    embeddings
)
hybrid = HybridRetriever(
    bm25,
    faiss
)
llm = LLMRecommender()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
def latest_user_message(conversation):
    for message in reversed(conversation):
        if message["role"] == "user":
            return message["content"]
    return ""

def build_context(conversation):
    lines = []
    for message in conversation:
        role = message["role"].capitalize()
        lines.append(
            f"{role}: {message['content']}"
        )
    return "\n".join(lines)

In [7]:
def build_search_query(conversation):

    user_messages = [
        message["content"]
        for message in conversation
        if message["role"] == "user"
    ]

    return " ".join(user_messages)

In [8]:
VAGUE_WORDS = {
    "assessment",
    "assessments",
    "test",
    "tests",
    "job",
    "role",
    "candidate",
    "hire",
    "hiring",
    "need"
}

def is_vague(query):
    q = query.lower()
    if (
        "engineer" in q or
        "developer" in q or
        "analyst" in q or
        "manager" in q or
        "graduate" in q or
        "intern" in q or
        "software" in q or
        "java" in q or
        "python" in q or
        "sql" in q
    ):
        return False
    words = q.split()
    if all(word in VAGUE_WORDS for word in words):
        return True
    return False

In [9]:
class ChatAgent:

    def __init__(self, retriever, embedder, llm):
        self.retriever = retriever
        self.embedder = embedder
        self.llm = llm

    def chat(self, conversation):
        context = build_context(conversation)
        query = build_search_query(conversation)
        if is_vague(query):
            return {
                "reply":
                "Could you tell me the role, seniority level, or important skills you're hiring for?",
                "recommendations":[]
            }
        query_embedding = self.embedder.encode([query])
        results = self.retriever.search(
            query=query,
            query_embedding=query_embedding,
            top_k=5
        )
        assessment_text = ""
        for result in results:
            assessment_text += f"""
Name: {result['name']}
Assessment Type: {result['assessment_type']}
Job Level: {result['job_level']}
Duration: {result['duration']}
URL: {result['url']}
"""
        explanation = self.llm.recommend(
            query=context,
            assessments=assessment_text
        )
        return {
            "reply": explanation,
            "recommendations": results
        }

In [10]:
agent = ChatAgent(
    hybrid,
    embedder,
    llm
)

In [11]:
conversation = [
    {
        "role":"user",
        "content":"Need assessment"
    },
    {
        "role":"assistant",
        "content":"Which role?"
    },
    {
        "role":"user",
        "content":"Graduate Java Engineer"
    }
]

In [12]:
response = agent.chat(conversation)

ResponseError: llama-server reported out-of-memory during startup: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 184541440
alloc_tensor_range: failed to allocate CUDA_Host buffer of size 184541440
error loading model: unable to allocate CUDA_Host buffer (status code: 500)

In [ ]:
print(response["reply"])

Here's a brief explanation of why each assessment is relevant for a Graduate Java Engineer:

1. **Java 8 (New)**: Assesses fundamental knowledge of Java 8 features, ensuring the candidate can write efficient and effective code.

2. **Java Design Patterns (New)**: Evaluates understanding of design patterns in Java, demonstrating the ability to solve complex problems and design robust software systems.

3. **Core Java (Advanced Level) (New)**: Tests advanced concepts in core Java, such as multithreading, networking, and exception handling, to assess the candidate's mastery of fundamental Java principles.

4. **Java Platform Enterprise Edition 7 (Java EE 7)**: Assesses knowledge of Java EE 7 features, verifying the candidate's ability to develop scalable, enterprise-level applications using Java EE.

5. **Java Frameworks (New)**: Evaluates familiarity with popular Java frameworks, such as Spring or Hibernate, and demonstrates the candidate's ability to leverage these frameworks for effici

In [ ]:
for item in response["recommendations"]:
    print("="*80)
    print(item["name"])
    print(item["assessment_type"])
    print(item["job_level"])
    print(item["duration"])

Java 8 (New)
Knowledge & Skills
Mid-Professional, Professional Individual Contributor
18 minutes
Java Design Patterns (New)
Knowledge & Skills
Mid-Professional, Professional Individual Contributor
5 minutes
Core Java (Advanced Level) (New)
Knowledge & Skills
Mid-Professional, Professional Individual Contributor
13 minutes
Java Platform Enterprise Edition 7 (Java EE 7)
Knowledge & Skills
Mid-Professional, Professional Individual Contributor
30 minutes
Java Frameworks (New)
Knowledge & Skills
Mid-Professional, Professional Individual Contributor
17 minutes


In [ ]:
conversation = [
    {
        "role":"user",
        "content":"Need assessment"
    },
    {
        "role":"assistant",
        "content":"Could you tell me the role?"
    },
    {
        "role":"user",
        "content":"Graduate Java Engineer"
    },
    {
        "role":"assistant",
        "content":"Here are some recommendations."
    },
    {
        "role":"user",
        "content":"Prefer shorter assessments under 30 minutes."
    }
]

In [ ]:
response = agent.chat(conversation)
print(response["reply"])

Here's a brief explanation for each recommended assessment:

1. **Java Platform Enterprise Edition 7 (Java EE 7)**:
Relevant for mid- to professional-level Java engineers, this assessment tests knowledge of enterprise Java development, including Java EE standards and technologies.

2. **Java 2 Platform Enterprise Edition 1.4 Fundamental**:
Suitable for entry- to mid-level developers or those looking to upskill, this assessment evaluates fundamental Java EE concepts and skills.

3. **Java 8 (New)**:
Ideal for professionals with a solid Java foundation, this 18-minute assessment assesses knowledge of new features in Java 8, such as lambda expressions and method references.

4. **Java Design Patterns (New)**:
This brief assessment is perfect for mid- to professional-level engineers who want to demonstrate their understanding of design patterns in Java. It's a great way to showcase expertise in software design principles.

5. **Core Java (Advanced Level) (New)**:
Targeted at mid- to profes